# Interpreting BERT Models (Part 1)

In this notebook we demonstrate how to interpret Bert models using  `Captum` library. In this particular case study we focus on a fine-tuned Question Answering model on SQUAD dataset using transformers library from Hugging Face: https://huggingface.co/transformers/

We show how to use interpretation hooks to examine and better understand embeddings, sub-embeddings, bert, and attention layers. 

Note: Before running this tutorial, please install `seaborn`, `pandas` and `matplotlib`, `transformers`(from hugging face, tested on transformer version `4.3.0.dev0`) python packages.

In [ ]:
!pip install datasets huggingface_hub 
!pip install git+https://github.com/huggingface/transformers
!pip install transformers==4.27.0
!pip install evaluate
!pip install lime

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from transformers import BertTokenizer, BertForQuestionAnswering, BertConfig

import torch
from torch.utils.data.dataloader import DataLoader

from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, EvalPrediction, GlueDataset
from transformers import AutoTokenizer, AutoModel, AutoModelWithLMHead
from transformers import GlueDataTrainingArguments as DataTrainingArguments


import lime

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

The first step is to fine-tune BERT model on SQUAD dataset. This can be easiy accomplished by following the steps described in hugging face's official web site: https://github.com/huggingface/transformers#run_squadpy-fine-tuning-on-squad-for-question-answering 

Note that the fine-tuning is done on a `bert-base-uncased` pre-trained model.

After we pretrain the model, we can load the tokenizer and pre-trained BERT model using the commands described below. 

In [ ]:
# replace <PATH-TO-SAVED-MODEL> with the real path of the saved model
model_path = 'randellcotta/distilbert-base-uncased-finetuned-yelp-polarity'

# load model
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()
model.zero_grad()

# load tokenizer
tokenizer=AutoTokenizer.from_pretrained(model_path)

Data loading and tokenization

In [ ]:
question_neg, text_neg = "Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff. It seems that his staff simply never answers the phone. It usually takes 2 hours of repeated calling to get an answer. Who has time for that or wants to deal with it? I have run into this problem with many other doctors and I just don't get it. You have office workers, you have patients with medical needs, why isn't anyone answering the phone? It's incomprehensible and not work the aggravation. It's with regret that I feel that I have to give Dr. Goldberg 2 stars.", '0'
question_pos, text_pos = "Before I finally made it over to this range I heard the same thing from most people - it's just fine to go work on your swing. I had such a low expectation I was pleasantly surprised. \n\nIt's a fairly big range - if you are familiar with Scally's in Moon, it seems like it has almost as many tees, though its not nearly as nice a facility. \n\nThe guys in the pro shop were two of the friendlier guys I've come across at ranges or at courses. Yards were indeed marked and there are some targets to aim for, and even some hazards to aim away from. \n\nA big red flag to me was the extra charge ($3) to hit off the grass. I am no range expert, but this is the 4th one I've been to and the first I've seen of that sort of nickel and diming....\n\nPrice for the golf balls was reasonable and I do plan to be back every week until they close up in October for the season. Hopefully, since its for sale, it will reopen as a golf facility again.",'1'

In [ ]:
# print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
# print('Cached:   ', round(torch.cuda.memory_cached(0)/1024**3,1), 'GB')

In [ ]:
tokenized_stuff_neg=tokenizer.encode(question_neg,return_tensors='pt')
tokenized_stuff_pos=tokenizer.encode(question_pos,return_tensors='pt')

In [ ]:
# print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
# print('Cached:   ', round(torch.cuda.memory_cached(0)/1024**3,1), 'GB')

In [ ]:
with torch.no_grad():
    out_neg = model(input_ids=tokenized_stuff_neg.to(device))
    out_pos = model(input_ids=tokenized_stuff_pos.to(device))

In [ ]:
# print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
# print('Cached:   ', round(torch.cuda.memory_cached(0)/1024**3,1), 'GB')

In [ ]:
print("Negative Sample Prediction ", out_neg)
print("Positive Sample Prediction ", out_pos)

Negative Sample Prediction  SequenceClassifierOutput(loss=None, logits=tensor([[ 3.6363, -3.7741]]), hidden_states=None, attentions=None)
Positive Sample Prediction  SequenceClassifierOutput(loss=None, logits=tensor([[-2.1887,  1.9726]]), hidden_states=None, attentions=None)


Using LIME toolbox for LIME

In [ ]:
from lime.lime_text import LimeTextExplainer
explainer = LimeTextExplainer(class_names=['Negative','Positive'])

In [ ]:
# classfn_out_pos

In [ ]:
# print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
# print('Cached:   ', round(torch.cuda.memory_cached(0)/1024**3,1), 'GB')

In [ ]:
hello=tokenizer(question_pos,return_tensors='pt',padding=True).to(device)

In [ ]:
# print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
# print('Cached:   ', round(torch.cuda.memory_cached(0)/1024**3,1), 'GB')

In [ ]:
def fun_predict(question_pos):
  out_pos = model(**tokenizer(question_pos,return_tensors='pt',padding=True).to(device))
  out_pos=out_pos[0].to(device)
  classfn_out_pos=torch.softmax(out_pos,dim=1).detach().cpu().numpy()
  print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
  print('Cached:   ', round(torch.cuda.memory_cached(0)/1024**3,1), 'GB')
  return classfn_out_pos

In [ ]:
fun_predict(question_pos)

Allocated: 0.0 GB
Cached:    0.0 GB


/usr/local/lib/python3.9/dist-packages/torch/cuda/memory.py:416: FutureWarning: torch.cuda.memory_cached has been renamed to torch.cuda.memory_reserved
  warnings.warn(


array([[0.0153486 , 0.98465145]], dtype=float32)

In [ ]:
exp = explainer.explain_instance(question_pos,fun_predict, num_samples=250,num_features=10)

NameError: ignored

In [ ]:
# print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
# print('Cached:   ', round(torch.cuda.memory_cached(0)/1024**3,1), 'GB')

In [ ]:
exp.show_in_notebook(text=question_pos)